# Emission to image: a Fourier Neural Operator for Kerr lensing (GPU)

Learns what gravity does to the light of an accretion disk. The input is a disk
emission profile as the camera would see it with gravity switched off (a plain
projection of the disk), plus the black hole's spin and inclination and the pixel
coordinates. The output is the image the ray tracer produces: the far side of the
disk lifted over the shadow, the photon ring, the shadow, and the g^4 Doppler and
gravitational shift.

It trains an FNO and a U-Net baseline (matched parameter count) at 64x64, then
tests both at 64, 128 and 256: the resolution-transfer question is whether a model
trained cheaply on coarse images holds up on fine ones, where ray tracing is
expensive. It ends with a speed comparison against the ray tracer.

**Before running:** Settings -> Accelerator -> **GPU T4**, Internet **on**, and
**Add Data -> kerr-lensing** (the 128x128 dataset; the 64x64 and 256x256 sets are
traced here, a few minutes on the GPU).

In [ ]:
# The repo goes to /tmp, so the notebook's output holds only the results.
import os, shutil, sys
REPO_URL = "https://github.com/amark-23/black-hole-ml.git"
REPO = "/tmp/black-hole-ml"
if os.path.exists(REPO):
    shutil.rmtree(REPO)
!git clone -q --depth 1 {REPO_URL} {REPO}
!pip -q install -e {REPO}
for sub in ("ml", "ml/datagen", "simulation/viz"):     # bhml, the generator, the ray tracer
    sys.path.insert(0, os.path.join(REPO, sub))

import glob, json, time
import numpy as np
import torch
import lensing                                   # ml/datagen/lensing.py
from bhml import fno_lensing as F
from bhml.lensing_data import load_lensing, fixed_emissions
from bhml.models import count_params

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (enable the GPU)")
OUT = "/kaggle/working/fno_results"
os.makedirs(OUT, exist_ok=True)

## Settings

`n_holes` and `seed` must match the published dataset (2000, 0): they fix which
black holes exist, so every resolution has the same ones, hole for hole.

In [ ]:
CFG = dict(
    n_holes=2000, seed=0,
    train_res=64, eval_res=(64, 128, 256),
    epochs=150, batch_size=32, lr=1e-3, weight_decay=1e-4, lr_step=50, lr_gamma=0.5,
    fno=dict(modes=12, width=32, depth=4, pad_frac=0.125),
    unet=dict(width=34, n_levels=3),       # about the FNO's parameter count
)
DATA_IN = "/kaggle/input"                   # where Add Data mounts the published set
DATA_WORK = "/tmp/kerr_lensing"             # sets traced here

## Data

The 128x128 test holes come from the published dataset. The 64x64 set (all
splits, for training) and the 256x256 test holes are traced here with the same
generator, seed and framing, so the same black holes appear at every resolution.

In [ ]:
def published(res):
    """Folder of the attached kerr-lensing dataset at this resolution, if any."""
    for m in glob.glob(f"{DATA_IN}/**/manifest.json", recursive=True):
        man = json.load(open(m))
        if man.get("name") == "kerr_lensing" and man.get("res") == res:
            return os.path.dirname(m)

def traced_here(res, splits=None):
    out = f"{DATA_WORK}_{res}" + ("_" + "_".join(splits) if splits else "")
    if not os.path.exists(os.path.join(out, "manifest.json")):    # reuse on a re-run
        t = time.time()
        lensing.generate(out, n=CFG["n_holes"], res=res, seed=CFG["seed"], device=device,
                         splits=splits, progress=lambda s: None)
        print(f"traced {res}x{res} ({'all' if not splits else ','.join(splits)}) in {time.time() - t:.0f}s")
    return out

R = CFG["train_res"]
full = load_lensing(traced_here(R))                       # every hole, for training
train, val = full.subset("train"), full.subset("val")
tests = {R: full.subset("test")}
for res in CFG["eval_res"]:
    if res == R:
        continue
    src = published(res)
    if src:
        tests[res] = load_lensing(src).subset("test")
        # The fixed ray tracer leaves ~0.05% of pixels unfinished; the old one ~1% on face-on holes.
        unfinished = float((tests[res].otype == 0).mean())
        print(f"{res}x{res} test holes from the published dataset: {100 * unfinished:.3f}% unfinished pixels")
        if unfinished > 0.0015:
            print("WARNING: this looks like the kerr-lensing version made before the ray-tracer "
                  "fix; regenerate it with dataset_lensing_kaggle.ipynb and attach the new version.")
    else:
        tests[res] = load_lensing(traced_here(res, ("test",)))

# The same test holes at every resolution, with the same parameters:
for res, t in tests.items():
    assert np.array_equal(t.index, tests[R].index), res
    assert np.allclose(t.params, tests[R].params), res
print(f"train {len(train)}, val {len(val)}, test {len(tests[R])} holes at {sorted(tests)}")

emis = fixed_emissions(CFG["n_holes"])      # one fixed profile per hole, for val and test

## Training pairs

Top row: the input (an emission profile seen with gravity off). Bottom row: the
target (the same emission through the ray-traced geometry). Training draws a fresh
random profile for every hole every epoch, so 1,600 black holes give unlimited pairs.

In [ ]:
import matplotlib.pyplot as plt
from bhml.lensing_data import make_batch
rows = np.argsort(tests[R].params[:, 1])[[0, 50, 100, 150, 199]]      # face-on to edge-on
x, y, _ = make_batch(tests[R], rows, emis.take(torch.from_numpy(tests[R].index[rows])), device)
fig, ax = plt.subplots(2, len(rows), figsize=(2.6 * len(rows), 5.4))
for j, r in enumerate(rows):
    a, inc = tests[R].params[r]
    ax[0, j].imshow(x[j, ..., 0].cpu() ** 0.5, cmap="inferno"); ax[0, j].set_title(f"a={a:.2f}, i={inc:.0f}°", fontsize=9)
    ax[1, j].imshow(y[j, ..., 0].cpu().clamp(min=0) ** 0.5, cmap="inferno")
ax[0, 0].set_ylabel("input: gravity off"); ax[1, 0].set_ylabel("target: ray traced")
for a_ in ax.ravel(): a_.set_xticks([]); a_.set_yticks([])
plt.tight_layout(); plt.savefig(f"{OUT}/pairs.png", dpi=110); plt.show()

## Train

Both models see the same data, loss (relative L2), optimizer and schedule; the
epoch with the best validation error is kept.

In [ ]:
models, hists = {}, {}
for kind, name in [("fno", "FNO"), ("unet", "U-Net")]:
    models[name], hists[name] = F.train_model(kind, train, val, emis, CFG, device)
params = {name: count_params(m) for name, m in models.items()}
print({k: f"{v / 1e6:.2f}M" for k, v in params.items()})
F.plot_history(hists, f"{OUT}/training.png")

## Accuracy and resolution transfer

Test error at the training resolution and on finer grids. The FNO's weights act on
Fourier modes, so it runs unchanged on any grid; the U-Net's 3x3 kernels each cover
a smaller patch of sky on a finer grid.

In [ ]:
errors = {name: {res: float(F.evaluate(m, tests[res], emis, device, batch_size=16).mean())
                 for res in CFG["eval_res"]} for name, m in models.items()}
print(f"{'model':8s}" + "".join(f"{res:>10d}²" for res in CFG["eval_res"]))
for name, e in errors.items():
    print(f"{name:8s}" + "".join(f"{100 * e[res]:10.2f}%" for res in CFG["eval_res"]))
F.plot_transfer(errors, CFG["train_res"], f"{OUT}/resolution_transfer.png")
ex = np.argsort(tests[R].params[:, 1])[[10, 100, 190]]       # face-on, mid, edge-on
for res in (R, max(tests)):
    F.plot_examples(models, tests[res], ex, emis, device, f"{OUT}/examples_{res}.png")
from IPython.display import Image, display
for f in ("training", "resolution_transfer", f"examples_{R}", f"examples_{max(tests)}"):
    display(Image(f"{OUT}/{f}.png"))

## Speed against the ray tracer

Seconds per image. The models' time includes building their inputs from (a,
inclination) and the emission; the ray tracer's is the batched GPU tracer the
dataset was made with.

In [ ]:
speed = {}
for res in CFG["eval_res"]:
    t = tests[res].params[:64]
    speed[res] = {"ray tracer": F.time_ray_tracer(t, res, device)}
    for name, m in models.items():
        speed[res][name] = F.time_model(m, res, device, batch_size=16, manifest=tests[res].manifest)
print(f"{'res':>5s} {'ray tracer':>12s} {'FNO':>10s} {'U-Net':>10s} {'FNO speedup':>12s}")
for res, s in speed.items():
    print(f"{res:5d} {1e3 * s['ray tracer']:10.1f}ms {1e3 * s['FNO']:8.2f}ms {1e3 * s['U-Net']:8.2f}ms "
          f"{s['ray tracer'] / s['FNO']:11.0f}x")

In [ ]:
results = dict(cfg=CFG, params=params, errors=errors,
               speed_s_per_image={str(r): v for r, v in speed.items()},
               histories=hists, n_train=len(train), n_val=len(val), n_test=len(tests[R]))
json.dump(results, open(f"{OUT}/results.json", "w"), indent=2)
for name, m in models.items():
    torch.save(m.state_dict(), f"{OUT}/{name.lower().replace('-', '')}.pt")
print("saved to", OUT, ":", sorted(os.listdir(OUT)))

## Notes

- **The input.** The emission profile is shown to the models as the camera would
  see it with gravity off, a straight-line projection of the disk onto the same
  pixels. Input and output then live on the same image grid at every resolution,
  and what the models learn is purely the relativistic part.
- **Emission profiles** are random smooth positive functions: a power law in r
  with streaks, spirals and hot spots (`bhml.lensing_data.random_emissions`),
  fresh for every hole every epoch, fixed per hole for validation and testing.
- **Errors** are relative L2 over the whole image, per test hole, averaged.
- **The 256x256 truth** is traced with the same generator, seed and framing as the
  published 128x128 set, for the test holes only.